## 1. Enter the deployed API URL

In [ ]:
import json
from contextlib import closing
from getpass import getpass
from http.client import HTTPConnection, HTTPSConnection
from urllib.parse import urlsplit

api = urlsplit(input('API base URL [https://api.domrov.app]: ').strip() or 'https://api.domrov.app')
local = api.hostname in {'localhost', '127.0.0.1', '::1'}
if (not api.hostname or api.username is not None or api.path not in {'', '/'}
        or api.query or api.fragment
        or not (api.scheme == 'https' or (local and api.scheme == 'http'))):
    raise ValueError('Enter an HTTPS API origin or a local HTTP origin.')


def request(path, expected, *, token='', body=None):
    connection = HTTPSConnection if api.scheme == 'https' else HTTPConnection
    headers = {'Accept': 'application/json'}
    if token:
        headers['Authorization'] = f'Bearer {token}'
    if body is not None:
        headers['Content-Type'] = 'application/json'
    with closing(connection(api.hostname, api.port, timeout=10)) as client:
        client.request('POST' if body is not None else 'GET', path,
                       json.dumps(body).encode() if body is not None else None, headers)
        response = client.getresponse()
        status = response.status
        request_id = response.getheader('X-Request-ID')
    print(f'HTTP {status} | X-Request-ID: {request_id or "missing"}')
    if status != expected or not request_id:
        raise RuntimeError(f'Expected HTTP {expected} with an X-Request-ID from Auth.')


## 2. Request a missing Auth route — 404

In [ ]:
request('/auth/demo-route-not-found', 404)

## 3. Request admin users as a regular user — 403

In [ ]:
request('/admin/users', 403, token=getpass('Regular demo account access token: '))

## 4. Reproduce a missing stored password hash — 500

In [ ]:
email = input('Prepared demo account email (Enter to skip): ').strip()
if email:
    request('/auth/login', 500,
            body={'email': email, 'password': getpass('Demo password: ')})
